In [1]:
import pyterrier as pt
import pandas as pd
import ir_datasets
import torch
from tqdm import tqdm
from transformers import AutoTokenizer

from data import MsMarcoTripletDataset, MsMarcoCollator, SPLADEArgs
from model import SPLADEModel
from trainer import IRTrainer

/opt/miniconda3/envs/splade/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = MsMarcoTripletDataset()

Building doc_store...


doc_store: 100%|██████████| 8841823/8841823 [00:29<00:00, 297508.02it/s]


Docstore built!
Building query_store...


query_store: 100%|██████████| 502939/502939 [00:03<00:00, 140478.85it/s]


Querystore built!
Building doc_pair_store...


doc_pair_store:   0%|          | 100000/269919004 [00:00<20:29, 219531.67it/s]

Doc_pair_store built!


In [3]:
args = SPLADEArgs("./training_ouput_dir/")

args.tokenizer_name_or_path = "distilbert-base-uncased"
args.model_name_or_path = "distilbert-base-uncased" 
args.shared_weights = True
args.n_negatives = 1
args.splade_doc = False
args.model_q = None
args.dense = False
args.full_determinism = False, 
args.seed = 123
args.lr = 2e-5
args.seed = 123
args.gradient_accumulation_steps = 1
args.weight_decay = 0.01
args.validation_metrics = [ "MRR@10", "recall@100", "recall@200", "recall@500" ]
args.pretrained_no_yamlconfig = False
args.nb_iterations = 150000
args.per_device_train_batch_size = 32  # number of gpus needs to divide this
args.per_device_eval_batch_size = 32
args.index_retrieve_batch_size = 500
args.record_frequency = 10000
args.train_monitoring_freq = 50
args.warmup_steps = 6000
args.max_length = 256
args.fp16 = True
args.matching_type = "splade"
args.monitoring_ckpt = "MRR@10"  # or e.g. MRR@10
args.lexical_type = "none"
args.training_loss = "contrastive"
args.T_d = 0
args.T_q = 0
args.l0d = 5e-4
args.l0q = 5e-4
args.top_d = -1
args.top_q = -1
args.evaluation_strategy = "epoch"
args.do_eval = True
args._flops = 1 # fix
args.report_to = ["wandb"]
args.save_steps = 1000

In [4]:
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name_or_path)
collator = MsMarcoCollator(tokenizer = tokenizer)

In [5]:
model = SPLADEModel(args.model_name_or_path,shared_weights=args.shared_weights,n_negatives=args.n_negatives, tokenizer=tokenizer, splade_doc=args.splade_doc, model_q=args.model_q)

In [6]:
trainer = IRTrainer(model = model,                         
                    args = args,                  
                    train_dataset = ds,
                    data_collator = collator.torch_call,
                    tokenizer = tokenizer,
                    shared_weights = args.shared_weights,  
                    splade_doc = args.splade_doc,          
                    n_negatives = args.n_negatives,         
                    dense = args.dense,
                   )

Using amp half precision backend


In [7]:
trainer.train()

/opt/miniconda3/envs/splade/lib/python3.8/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 100000
  Num Epochs = 3
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 9375
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: jackmckechnie (jack-terrier). Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("splade_model")

In [8]:
model.save("splade_model", tokenizer = tokenizer)

Configuration saved in splade_model/config.json
tokenizer config file saved in splade_model/tokenizer_config.json
Special tokens file saved in splade_model/special_tokens_map.json


In [8]:
# Sensitivity SPLADE dataset
import pandas as pd
import numpy as np
import re
import string



ohsumed_training_docs = get_d2qmm()

In [ ]:
class OHSUMEDDataset():
    
    def __init__(self):
        print("Loading docs")
        
        ohsumed_training_docs = self.get_d2qmm()
        
        sensitive_docs = ohsumed_training_docs[ohsumed_training_docs["sensitivity"] == 1].reset_index()
        non_sensitive_docs = ohsumed_training_docs[ohsumed_training_docs["sensitivity"] == 0].head(len(sensitive_docs)).reset_index()
        self.len = len(non_sensitive_docs)

        self.sensitive_docs = {index: f"{row['text']}" for index, row in sensitive_docs.iterrows()}
        self.non_sensitive_docs = {index: f"{row['text']}" for index, row in non_sensitive_docs.iterrows()}
        self.queries = {index: f"{row['querygen']}" for index, row in sensitive_docs.iterrows()}
        print("Docs loaded")

    def get_d2qmm(self):
        d2qmm = pd.read_pickle("/nfs/primary/sas_cross_encoder/d2qmm_ohsumed_10samples_0.1filter.pkl")
        max_score_indices = []
        for row_scores in d2qmm['querygen_score']:
            if len(row_scores) > 0:
                max_score_indices.append(np.argmax(np.array(row_scores)))
            else:
                # Handle the case of an empty list
                max_score_indices.append(None)
    
        # Update 'querygen' to keep only the query with the top score
        d2qmm['querygen'] = [queries.split('\n')[index] if (index is not None and queries) else None for queries, index in zip(d2qmm['querygen'], max_score_indices)]
    
        # Drop the 'querygen_score' column as it's no longer needed
        d2qmm = d2qmm.drop('querygen_score', axis=1)
        d2qmm['querygen'] = d2qmm['querygen'].str.replace(f"[{re.escape(string.punctuation)}]", "", regex = True)
        
        return d2qmm
        
    def __len__(self):
        return self.len
        
    def __getitem__(self, idx):
        query = self.queries[idx]
        pos_doc = self.sensitive_docs[idx]
        neg_doc = self.non_sensitive_docs[idx]
        docs = [query, pos_doc, neg_doc]
        scores = torch.tensor([0, 0, 0]).view(1, -1)
        return docs, scores